# Dunnhumby seed 43 — M2 공유 N/V 표현 + M3 중심화 가치그래프

새 학습은 **부분결합 M5 하나**입니다. M2의 사용자 N·V와 상품 가격 특징을 모두 유지하고, 기존 M3 가치축 그래프에서 처음부터 하나의 optimizer·plain BPR로 공동학습합니다. M4 행가중, 동결, 사후 점수 합산은 없습니다. 신규상품 추천·MIN_ITEM_INTER=1·uniform K=1·개발구간 684–690일만 사용하며 최종 test/holdout은 열지 않습니다.

M1/M2는 검증된 M2 결과의 readout, M3는 기존 300-epoch 곡선을 **이번 조기종료 규칙으로 사후 재판독**한 readout입니다. M3의 원래 100-epoch 후보 판정과 섞지 않습니다. 세 원본이 없거나 불일치하면 학습 전에 중단하며 자동 재학습하지 않습니다. 원본 간 M1의 125-epoch 노출·가격 7항목과 M3 감사 순위상관 2항목 차이는 숨기지 않고 기록합니다.

최대300 epoch, 25마다 개발평가, 100 이후 4회 미개선 시 중단·최적점 복원. 질문은 결합이 재판독 M3보다 전체 가격·구매금액 가중 적중값@10과 가중 NDCG@10을 함께 높이는지입니다. 동시에 전체 Recall/NDCG @10/20/50 각각 M1의 99% 이상이어야 합니다. 단일 반복노출 개발시드라 유의성·재현성·CLV 귀속·시너지는 주장할 수 없습니다.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
from google.colab import drive
ROOT = Path('/content/drive/MyDrive/논문/data')
M2_REPORT = ROOT/'results_v3_dunnhumby_shared_feature_residual_m2_seed43_v1/reports/result.json'
M3_REPORT = ROOT/'results_v3_dunnhumby_clv_m3_centered_value_graph_v1/clv_m3_centered_value_graph_244783a20cc5.json'
OUT = ROOT/'results_v3_dunnhumby_shared_feature_centered_graph_seed43_v1'
try:
    if not ROOT.is_dir():
        drive.mount('/content/drive')
    if not ROOT.is_dir():
        raise RuntimeError(f'Drive 자료 폴더를 찾지 못했습니다: {ROOT}')
    for label, path in [('M2', M2_REPORT), ('M3', M3_REPORT)]:
        if not path.is_file():
            raise RuntimeError(f'{label} 원본 JSON이 없습니다: {path}. 경로만 확인해 수정하세요. 학습은 시작되지 않았습니다.')
except (OSError, ValueError, NotImplementedError) as exc:
    raise RuntimeError('Drive 연결/읽기 실패. 마운트 상태를 확인하고 이 셀부터 다시 실행하세요. 학습은 시작되지 않았습니다.') from exc
SOURCE_COMMIT = '93c9492aa0cca4454ad1e4c36719dde9512ad8df'
REPO = Path('/content/clv-m2m3-joint-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
HEAD = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
assert HEAD == SOURCE_COMMIT, '코드 버전이 다릅니다.'
for name in ('lightgcn_clv_v3', 'clv_shared_feature_residual_m2_screen', 'lightgcn_clv_m3_centered_value_graph'):
    if name in sys.modules:
        assert Path(sys.modules[name].__file__).resolve().parent == REPO.resolve(), '다른 실험 코드가 로드되어 있습니다. 런타임을 다시 시작하세요.'
os.chdir(REPO)
sys.path.insert(0, str(REPO))
import clv_shared_feature_centered_graph_screen as screen
print('고정 코드:', HEAD)


## 1. 원본·입력·그래프 확인 (GPU 학습 없음)
해시·학습조건·신규상품 분할·M3 가중치 게이트를 검증하고, M1/M2/M3의 선택점을 출력합니다. 이 단계에서 오류가 나면 다음 학습 셀을 실행하지 마세요.

In [ ]:
cfg, prepared = screen.prepare(M2_REPORT, M3_REPORT, OUT)
assert cfg.seeds == (43,) and cfg.epochs == 300 and cfg.negative_count == 1
assert [a['model_id'] for a in prepared['anchors']] == ['m1', screen.m2.MODEL_ID, screen.m3.ARM_VALUE]
assert not screen._spec()['weighted']
print('기준 선택점:', [(a['model_id'], a['selected_epoch'], a['stopped_epoch']) for a in prepared['anchors']])
print('학습 조건:', screen.m2.selection(cfg))
print('M3 엣지 감사:', json.dumps(prepared['graph_audit'], ensure_ascii=False, indent=2))


## 2. 부분결합 하나 학습
GPU가 필요합니다. 완료 epoch마다 모델·optimizer·난수상태를 Drive에 저장하므로, 중단되면 같은 노트북을 다시 실행해 이어갑니다. N/V 초기 추천손실 기울기가 없으면 학습 전에 중단합니다.

In [ ]:
import torch
assert torch.cuda.is_available(), '학습에는 GPU 런타임을 사용하세요.'
paths = screen.run(cfg, prepared)
print(json.dumps(paths, ensure_ascii=False, indent=2))


## 3. 결과 확인·다운로드
전체 142지표와 불리한 결과를 함께 확인하세요. ZIP은 표·진단·판정 JSON만 담고, 재개용 checkpoint는 Drive에 남깁니다.

In [ ]:
import pandas as pd
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
report = json.loads((OUT/'reports/result.json').read_text())
assert report['code_version'] == screen.VERSION and report['new_fit_count'] == 1
print('M1 / M2 / M3 / M2+M3 전체 절대지표 (선택점)')
print(pd.read_csv(report['paths']['absolute']).set_index('model_id').T.to_string())
print('판정:', json.dumps(report['reading'], ensure_ascii=False, indent=2))
print('전체 비교·곡선·진단:', {k: v for k, v in report['paths'].items() if k != 'absolute'})
zip_path = Path('/content/shared_feature_centered_graph_seed43_results.zip')
with ZipFile(zip_path, 'w', compression=ZIP_DEFLATED) as archive:
    for path in report['paths'].values():
        archive.write(path, arcname=Path(path).name)
files.download(str(zip_path))
